In [1]:
# ===== Célula 1 — Carregar o buscador =====
import sys
import time
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.rag.recuperacao import Buscador

pd.set_option("display.max_colwidth", 90)
inicio = time.perf_counter()
buscador = Buscador()
print(f"Buscador pronto em {time.perf_counter() - inicio:.0f}s")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Buscador pronto em 8s


In [2]:
# ===== Célula 2 — Comparar as três buscas lado a lado =====
pergunta = "Os produtos chegam com defeito ou quebrados?"

densa = buscador.busca_densa(pergunta).head(8).join(
    buscador.corpus["texto"], on="review_id")
lexical = buscador.busca_bm25(pergunta).head(8).join(
    buscador.corpus["texto"], on="review_id")
hibrida = buscador.buscar(pergunta, k_final=8)

print("SENTIDO (Chroma):");  display(densa[["pos_densa", "sim_densa", "texto"]])
print("PALAVRAS (BM25):");   display(lexical[["pos_bm25", "score_bm25", "texto"]])
print("HÍBRIDA (RRF + colapso):")
display(hibrida[["ref", "pos_densa", "pos_bm25", "rrf", "repeticoes", "nota", "texto"]])

SENTIDO (Chroma):


,pos_densa,sim_densa,texto
0,1,0.915514,"Negativo. O produto realmente foi entregue no prazo, entretanto chegou todo quebrado c..."
1,2,0.913442,Produto com defeito. Produto veio quebrado
2,3,0.912701,Produto mal embalado. Produto chegou quebrado é material não é bom
3,4,0.911992,"O Primeiro produto veio com defeito. Depois que reclamei enviaram outro rapidamente, e..."
4,5,0.911409,falta de produto e quebra. Faltou 1 produto e os que recebi 1 veio quebrado
5,6,0.909759,"Embalagem. Produto de boa qualidade , entrega rápida, infelizmente não foi bem embalad..."
6,7,0.909026,"Produto entregue não condiz com o produto comprado, péssima qualidade, assessório veio..."
7,8,0.908627,2. A caixa veio totalmente deformada/estragada. Alguns produtos desencaixados. Melhora...


PALAVRAS (BM25):


,pos_bm25,score_bm25,texto
0,1,5.152988,Produtos baratos e chegam bem antes do prazo.
1,2,5.000056,"OTIMA, OS PRODUTOS CHEGAM PERFEITOS E SEMPRE NO PRAZO, A UNICA COISA QUE ACHO CARO E O..."
2,3,4.759727,estou satisfeito com os produtos
3,4,4.739877,Parabéns a lannister.com e toda à sua equipe ! Os produtos chegam sempre antes do praz...
4,5,4.725626,Sempre fico satisfeita com o atendimentatendimento e/ou produtos.
5,6,4.714261,Alguns cabides chegaram quebrados
6,7,4.642968,"tablets quebrados. alguns tablets de bcaa vieram quebrados, diminuindo a qualidade do ..."
7,8,4.542743,Preciso que me reembolsem ou troquem pq está com defeito!


HÍBRIDA (RRF + colapso):


,ref,pos_densa,pos_bm25,rrf,repeticoes,nota,texto
0,R1,24.0,12.0,0.025794,1,2,Produto com defeito. produto com defeito
1,R2,NaN,1.0,0.016393,1,5,Produtos baratos e chegam bem antes do prazo.
2,R3,1.0,NaN,0.016393,1,1,"Negativo. O produto realmente foi entregue no prazo, entretanto chegou todo quebrado c..."
3,R4,NaN,2.0,0.016129,1,5,"OTIMA, OS PRODUTOS CHEGAM PERFEITOS E SEMPRE NO PRAZO, A UNICA COISA QUE ACHO CARO E O..."
4,R5,2.0,NaN,0.016129,1,3,Produto com defeito. Produto veio quebrado
5,R6,NaN,3.0,0.015873,1,5,estou satisfeito com os produtos
6,R7,3.0,NaN,0.015873,1,3,Produto mal embalado. Produto chegou quebrado é material não é bom
7,R8,4.0,NaN,0.015625,1,5,"O Primeiro produto veio com defeito. Depois que reclamei enviaram outro rapidamente, e..."


In [3]:
# ===== Célula 3 — Filtro + colapso de repetidos em ação =====
pergunta = "O que os clientes mais elogiam?"
filtros = {"nota": {"$gte": 4}}

resultado = buscador.buscar(pergunta, filtros=filtros, k_final=10)
print(f"Avaliações no recorte do filtro: {resultado.attrs['documentos_no_filtro']:,}")
display(resultado[["ref", "nota", "repeticoes", "categoria_principal", "texto"]])

Avaliações no recorte do filtro: 27,791


,ref,nota,repeticoes,categoria_principal,texto
0,R1,5,1,esporte lazer,"Confiança, ,. Adoro comprar aqui. Tenho o que necessito, em prazo mínimo. É de se ress..."
1,R2,5,1,utilidades domesticas,"Excelente 👍🏽. Respeito ao cliente, qualidade, satisfação. Cliente Feliz!!!"
2,R3,5,1,perfumaria,Recomendo a todos os clientes
3,R4,5,1,utilidades domesticas,"Elogio. Muito satisfeita, produto de ótima quelidade, entrega rápida. Recomendo."
4,R5,5,1,informatica acessorios,sempre mantendo o compromisso com os clientes
5,R6,5,1,perfumaria,"Confiabilidade. Qualidade, ótimo preço, rapidez na entrega! Nota 1000000"
6,R7,5,1,construcao ferramentas jardim,Parabéns pelo compromisso com os clientes
7,R8,5,1,informatica acessorios,Excelente recomendo. Melhor tempo de entrega. Produto de excelente qualidade. Supera a...
8,R9,5,1,telefonia,Parabéns pelo atenção com os clientes... 😚
9,R10,5,1,fashion calcados,"Maravilha... Olha...fiquei mto satisfeito com a mercadoria, e recomendo com certeza, f..."


In [2]:
# ===== Célula 4 — Linha de base (densa) + colapso de repetidos =====
pergunta = "O produto chegou quebrado"
resultado = buscador.buscar(pergunta, k_final=10)  # modo="densa" é o padrão
print(f"Pergunta: '{pergunta}'")
display(resultado[["ref", "sim_densa", "repeticoes", "nota", "texto"]])

Pergunta: 'O produto chegou quebrado'


,ref,sim_densa,repeticoes,nota,texto
0,R1,0.946337,2,1,O produto chegou quebrado
1,R2,0.941865,1,3,Produto mal embalado. Produto chegou quebrado é material não é bom
2,R3,0.941087,1,2,Produto veio quebrado
3,R4,0.936575,1,1,o produto veio quebrado
4,R5,0.935691,1,1,"Produto chegou quebrado, ficaram de trocar"
5,R6,0.932797,1,1,O produto chegou com uns parte quebrada
6,R7,0.932007,1,2,produto chegou no prazo porem veio quebrado
7,R8,0.929834,1,1,Produto entregue danifica. O produto foi entregue quebrado. nem foi preciso abrir a em...
8,R9,0.929595,1,2,"Produto veio quebrado. Fiquei muito chateada, pois o produto veio quebrado! Um descaso..."
9,R10,0.928249,1,3,Produto com defeito. Produto veio quebrado


In [5]:
# ===== Célula 5 — Primeira resposta com citações =====
from IPython.display import Markdown

from src.rag.geracao import gerar_resposta

pergunta = "O que os clientes relatam sobre produtos que chegaram quebrados?"
evidencias = buscador.buscar(pergunta, k_final=20)
resultado = gerar_resposta(pergunta, evidencias, modelo="openai/gpt-oss-120b")

print(f"Pergunta: {pergunta}")
print(f"Modelo: {resultado['modelo']} | {resultado['segundos']}s | "
      f"tokens entrada/saída: {resultado['tokens_entrada']}/{resultado['tokens_saida']}")
print(f"Citações: {resultado['citacoes']} | Inválidas: {resultado['citacoes_invalidas'] or 'nenhuma'}")
display(Markdown(resultado["resposta"]))

Pergunta: O que os clientes relatam sobre produtos que chegaram quebrados?
Modelo: openai/gpt-oss-120b | 1.1s | tokens entrada/saída: 1890/367
Citações: [] | Inválidas: nenhuma


## Resposta
Os clientes relatam que produtos chegam quebrados principalmente por embalagens inadequadas ou frágeis, e que isso costuma gerar insatisfação e necessidade de troca ou devolução. Em alguns casos o atendimento foi rápido e resolveu o problema, mas a maioria das avaliações aponta falhas na proteção da mercadoria durante o transporte.

## O que os clientes relatam
- Produto chegou quebrado ou danificado ao receber (R2, R3, R4, R5, R6, R7, R8, R9, R10, R11, R12, R13, R15, R16, R17, R18, R19).  
- Embalagem insuficiente, sem aviso de fragilidade ou material de proteção inadequado (R3, R9, R10, R12, R15, R16, R20).  
- Dúvidas sobre responsabilidade do transporte ou da loja (R4, R12).  
- Necessidade de troca ou devolução devido ao dano (R3, R4, R13, R16, R17, R19).  
- Atendimento eficaz em alguns casos, com substituição rápida (R1, R2).  

## Interpretação
A frequência de relatos de produtos quebrados indica um problema recorrente de embalagem e manuseio, afetando a percepção de qualidade e confiança dos clientes. Embora haja exemplos de bom atendimento pós‑incidente, a maioria das avaliações destaca a necessidade de melhorar a proteção dos itens frágeis antes do envio.

## Limitações
A resposta se baseia apenas na amostra de 20 avaliações recuperadas e não reflete a frequência real de ocorrências em toda a base de dados.

In [2]:
# ===== Célula 6 — Comparar os dois redatores em 3 tipos de pergunta (prompt v2) =====
import time

from IPython.display import Markdown

from src.rag.geracao import gerar_resposta

testes = [
    ("específica", "O que os clientes relatam sobre produtos que chegaram quebrados?", None),
    ("com filtro", "Quais problemas aparecem nas avaliações?",
     {"categoria_principal": "cama mesa banho", "nota": {"$lte": 2}}),
    ("fora do escopo", "Qual foi o faturamento da Olist em 2018?", None),
]
modelos = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b"]

linhas, respostas = [], {}
for tipo, pergunta, filtros in testes:
    evidencias = buscador.buscar(pergunta, filtros=filtros, k_final=20)
    for modelo in modelos:
        r = gerar_resposta(pergunta, evidencias, modelo=modelo)
        respostas[(tipo, modelo)] = (pergunta, filtros, r["resposta"])
        linhas.append({"tipo": tipo, "modelo": modelo.split("/")[-1],
                       "segundos": r["segundos"], "tokens_saida": r["tokens_saida"],
                       "n_citacoes": len(r["citacoes"]),
                       "citacoes_invalidas": len(r["citacoes_invalidas"]),
                       "recusou": r["recusou"],
                       "termos_frequencia": ", ".join(r["termos_frequencia"]) or "—"})
        time.sleep(5)  # pausa curta para respeitar o limite por minuto da camada gratuita

display(pd.DataFrame(linhas))

Limite de uso atingido; aguardando 20s (tentativa 1/3)
Limite de uso atingido; aguardando 20s (tentativa 1/3)


,tipo,modelo,segundos,tokens_saida,n_citacoes,citacoes_invalidas,recusou,termos_frequencia
0,específica,gpt-oss-120b,1.6,539,9,0,False,"frequentemente, maioria, principalmente"
1,específica,qwen3.8-27b,1.7,542,9,0,False,frequentemente
2,com filtro,gpt-oss-120b,1.8,648,16,0,False,—
3,com filtro,qwen3.8-27b,1.6,535,17,0,False,recorrentes
4,fora do escopo,gpt-oss-120b,0.5,90,0,0,True,—
5,fora do escopo,qwen3.8-27b,0.6,86,0,0,True,—


In [3]:
# ===== Célula 7 — Ler as respostas lado a lado =====
for (tipo, modelo), (pergunta, filtros, texto) in respostas.items():
    filtro_txt = f"  \n**Filtros:** `{filtros}`" if filtros else ""
    display(Markdown(f"---\n### [{tipo}] {modelo}\n**Pergunta:** {pergunta}{filtro_txt}\n\n{texto}"))

---
### [específica] openai/gpt-oss-120b
**Pergunta:** O que os clientes relatam sobre produtos que chegaram quebrados?

## Resposta
Os clientes relatam que produtos chegam quebrados principalmente por embalagens inadequadas ou frágeis, apontando falta de proteção e sinalização de fragilidade. Em alguns casos, o dano é atribuído ao transporte ou ao carteiro, mas a maioria menciona a própria embalagem como causa. Quando o problema ocorre, o atendimento costuma ser elogiado por resolver rapidamente, embora alguns clientes ainda expressem insatisfação e solicitem troca ou devolução.

## O que os clientes relatam
- **Embalagem insuficiente ou frágil:** “Produto mal embalado. Produto chegou quebrado” [R3]; “cuba chegou aqui toda quebrada, numa cx de papelao fininha sem proteçao” [R12]; “caixa veio totalmente deformada/estragada” [R10].
- **Ausência de aviso de fragilidade:** “não tinha especificação na caixa falando q era produto frágil” [R16]; “sem sinalização de produto frágil” implícito em R16.
- **Dano atribuído ao transporte ou carteiro:** “não sei se o dano ocorreu nos Correios ou se me enviado assim” [R4]; “não foi entregue por transport e sim pelo carteiro que nem sabia que era algo frágil” [R12].
- **Atendimento eficaz após reclamação:** “fui super bem atendida, imediatamente resolveram tudo e mandaram outro novinho” [R2]; “enviaram outro rapidamente, em perfeito estado” [R1].
- **Insatisfação e solicitação de troca ou devolução:** “não tô solicitando outro produto nem a devolução” [R4]; “Solicito troca urgente por um produto em boas condições” [R13]; “quero meu dinheiro de volta” [R15].

## Interpretação
Isso sugere que a principal causa percebida pelos clientes é a embalagem inadequada, especialmente a falta de proteção e sinalização de fragilidade, o que leva a danos durante o transporte. Embora o suporte ao cliente seja frequentemente elogiado por resolver o problema, a experiência inicial de receber produtos quebrados gera insatisfação e pedidos de troca ou reembolso.

## Limitações
A resposta se baseia em 20 avaliações recuperadas de um total de 42.251 avaliações com texto, e não mede a frequência dos relatos na base completa.

---
### [específica] qwen/qwen3.8-27b
**Pergunta:** O que os clientes relatam sobre produtos que chegaram quebrados?

## Resposta
No recorte de todas as avaliações com texto, os clientes relatam que produtos chegaram quebrados, frequentemente associando o dano a falhas na embalagem ou no transporte. Diversos relatos mencionam caixas amassadas, ausência de proteção adequada ou falta de identificação de "frágil" [R3, R9, R12, R16]. Alguns clientes também relatam que, após a reclamação, a empresa enviou substituições rapidamente [R1, R2].

## O que os clientes relatam
- **Embalagem inadequada:** Clientes citam embalagens frágeis ou ausentes, como "numa cx de papelao fininha sem proteçao" [R12] e "papel bolha marrom lotado de sombra" [R15].
- **Falta de identificação de frágil:** Há relatos de que os pacotes não tinham aviso de frágil, como "sem aviso de frágil, num papel pardo" [R9] e "não tinha especificação na caixa falando q era produto frágil" [R16].
- **Dano no transporte:** Alguns clientes atribuem a quebra ao transporte, citando "Pacote amassado nas pontas, peças quebradas" [R9] ou "caixa veio totalmente deformada/estragada" [R10].
- **Resolução pós-venda:** Relatos de que a empresa enviou produtos novos após a reclamação, como "enviaram outro rapidamente, em perfeito estado" [R1] e "imediatamente resolveram tudo e mandaram outro novinho" [R2].
- **Dúvida sobre a origem do dano:** Um cliente expressa incerteza sobre se o dano ocorreu no transporte ou na origem: "Não sei se o dano ocorreu nos Correios ou se me enviado assim" [R4].

## Interpretação
Isso sugere que a percepção de quebra está fortemente ligada à percepção de falha logística na proteção do item, mais do que à qualidade intrínseca do produto em si, embora alguns clientes também mencionem "material não é bom" [R3]. A rápida resolução em alguns casos parece mitigar a insatisfação inicial, levando a notas altas mesmo após o incidente [R1, R2].

## Limitações
A resposta usa 20 avaliações recuperadas de um recorte de 42.251 avaliações com texto e não mede frequências na base.

---
### [com filtro] openai/gpt-oss-120b
**Pergunta:** Quais problemas aparecem nas avaliações?  
**Filtros:** `{'categoria_principal': 'cama mesa banho', 'nota': {'$lte': 2}}`

## Resposta
No recorte de avaliações de cama mesa banho com nota ≤ 2, os principais problemas apontados são: produto com cor ou modelo diferente do anunciado [R1, R2, R12, R18]; baixa qualidade ou itens quebrados [R3, R4, R19]; falta de itens ou entrega incompleta [R5, R6, R17, R20]; atrasos e não‑entregas sem comunicação [R7, R8, R11, R16]; e dificuldades no acompanhamento do pedido pelo site [R10].

## O que os clientes relatam
- **Cor/modelo divergente:** “as cores vieram todas erradas” [R1]; “coloca uma foto recebe outro produto” [R2]; “recebi modelos diferentes… diferente do modelo mostrado no site” [R12]; “na foto é mais bonito… costuras engrugidas” [R18].
- **Qualidade ruim ou produto danificado:** “baixa qualidade… péssima qualidade” [R3]; “assessório veio quebrado” [R4]; “pano ruim… não é o que apresenta no site” [R19].
- **Falta de itens/entrega incompleta:** “Só entregou um item faltando os 2 outros” [R5]; “Veio faltando produto… só veio 1” [R6]; “faltando 02 panos de prato… paguei por 07” [R17]; “Só recebi um… não rastrear os 2 produtos” [R20].
- **Atrasos e não‑entregas sem informação:** “não chegou… imprevisto dos correios” [R7]; “produto não foi entregue… estou aguardando reembolso” [R8]; “não recebi… ainda tem muitos produtos que comprei” [R11]; “já se passaram mais de 30 dias… sem nenhum contato” [R16].
- **Problemas no acompanhamento pelo site:** “não pude acompanhar meu pedido pelo site… não atualiza” [R10].

## Interpretação
Essas reclamações sugerem que falhas de precisão nas informações de produto (cor, modelo, foto) e de controle de estoque geram entregas equivocadas, enquanto a baixa qualidade percebida e a ausência de itens aumentam a insatisfação. Além disso, atrasos e falta de comunicação sobre o status da entrega, bem como um site que não rastreia pedidos, agravam a percepção negativa.

## Limitações
A resposta se baseia em 20 avaliações recuperadas de um recorte de 1 144 avaliações com texto e não mede frequências na base.

---
### [com filtro] qwen/qwen3.8-27b
**Pergunta:** Quais problemas aparecem nas avaliações?  
**Filtros:** `{'categoria_principal': 'cama mesa banho', 'nota': {'$lte': 2}}`

## Resposta
As avaliações recuperadas, referentes ao recorte de categoria "cama mesa banho" com nota ≤ 2, apontam problemas recorrentes de divergência entre o produto anunciado e o recebido, baixa qualidade dos itens e falhas na entrega, como itens faltantes ou não entregues. Clientes relatam também dificuldades no pós-venda, como a impossibilidade de rastreamento e a lentidão ou ausência de contato da loja para resolver pendências.

## O que os clientes relatam
- **Divergência entre foto e produto:** Clientes afirmam que o item recebido não corresponde à imagem do site, incluindo cores erradas e modelos diferentes [R1, R2, R12, R18]. Um cliente cita: "coloca uma foto recebe outro produto" [R2].
- **Baixa qualidade do material:** Há relatos de produtos com "péssima qualidade", tecido fino e costuras ruins, não condizentes com a expectativa [R3, R4, R18, R19].
- **Itens faltantes ou não entregues:** Vários clientes relataram receber apenas parte do pedido ou nada, com registros de atraso ou não entrega [R5, R6, R7, R11, R16, R17, R20]. Um cliente afirma: "Comprei 2 cortinas e só veio 1" [R6].
- **Falhas no rastreamento e comunicação:** Clientes não conseguem acompanhar o pedido no site e relatam falta de contato da loja após o prazo, gerando incerteza sobre o status [R10, R16, R20].
- **Dificuldade no pós-venda e trocas:** Relatos de dor de cabeça para realizar trocas, cancelamentos negados ou demora excessiva para resolução de problemas [R1, R13, R14, R19].

## Interpretação
Isso sugere que a experiência de compra é comprometida por uma lacuna entre a curadoria de conteúdo (fotos/descrições) e a realidade do estoque/qualidade, somada a fragilidades logísticas que impedem o cliente de ter visibilidade sobre o pedido. A ausência de comunicação proativa da loja em situações de falha agrava a percepção de descaso.

## Limitações
A resposta usa 20 avaliações recuperadas de um recorte de 1.144 avaliações com texto e não mede frequências na base.

---
### [fora do escopo] openai/gpt-oss-120b
**Pergunta:** Qual foi o faturamento da Olist em 2018?

## Resposta
As avaliações recuperadas não são suficientes para responder a esta pergunta.  
**Motivo:** Nenhuma das avaliações traz informações sobre faturamento ou dados financeiros da Olist.  
**O que esta base consegue responder:** Perguntas sobre experiências de compra dos clientes, como produto, entrega, atendimento e loja.

---
### [fora do escopo] qwen/qwen3.8-27b
**Pergunta:** Qual foi o faturamento da Olist em 2018?

## Resposta
As avaliações recuperadas não são suficientes para responder a esta pergunta.
**Motivo:** O contexto contém apenas relatos de experiência de clientes sobre produtos, entrega e atendimento, não incluindo dados financeiros corporativos como faturamento.
**O que esta base consegue responder:** Perguntas sobre a satisfação dos clientes com a qualidade dos produtos, a pontualidade da entrega e a experiência de compra na loja.

In [4]:
# ===== Célula 8 — Testar a verificação de trechos nas respostas já geradas (sem gastar API) =====
import importlib

import src.rag.geracao as geracao

importlib.reload(geracao)  # carrega a versão nova do arquivo sem reiniciar o kernel

for (tipo, modelo), (pergunta, filtros, texto) in respostas.items():
    evid = buscador.buscar(pergunta, filtros=filtros, k_final=20)  # mesma busca, mesmo resultado
    problemas = geracao.verificar_trechos(texto, evid)
    print(f"[{tipo}] {modelo}: {len(problemas)} trecho(s) não encontrado(s)")
    for p in problemas:
        print(f"    -> {p}")

[específica] openai/gpt-oss-120b: 2 trecho(s) não encontrado(s)
    -> sem sinalização de produto frágil
    -> Solicito troca urgente por um produto em boas condições
[específica] qwen/qwen3.8-27b: 0 trecho(s) não encontrado(s)
[com filtro] openai/gpt-oss-120b: 2 trecho(s) não encontrado(s)
    -> baixa qualidade… péssima qualidade
    -> Só recebi um… não rastrear os 2 produtos
[com filtro] qwen/qwen3.8-27b: 0 trecho(s) não encontrado(s)
[fora do escopo] openai/gpt-oss-120b: 0 trecho(s) não encontrado(s)
[fora do escopo] qwen/qwen3.8-27b: 0 trecho(s) não encontrado(s)


In [5]:
# ===== Célula 9 — Conferir à mão os trechos sinalizados (sem gastar API) =====
import re

for (tipo, modelo), (pergunta, filtros, texto) in respostas.items():
    evid = buscador.buscar(pergunta, filtros=filtros, k_final=20)
    sinalizados = set(geracao.verificar_trechos(texto, evid))
    if not sinalizados:
        continue
    textos = evid.set_index("ref")["texto"]
    for trecho, citacao in geracao.PADRAO_TRECHO.findall(texto):
        if trecho.strip() not in sinalizados:
            continue
        refs = re.findall(r"R\d+", citacao or "")
        print(f"\n[{tipo}] {modelo}")
        print(f"  TRECHO NA RESPOSTA: “{trecho.strip()}”  -> citado como {refs or 'sem citação'}")
        for r in refs:
            if r in textos.index:
                print(f"  TEXTO REAL DE {r}: {textos[r]}")


[específica] openai/gpt-oss-120b
  TRECHO NA RESPOSTA: “sem sinalização de produto frágil”  -> citado como sem citação

[específica] openai/gpt-oss-120b
  TRECHO NA RESPOSTA: “Solicito troca urgente por um produto em boas condições”  -> citado como ['R13']
  TEXTO REAL DE R13: Produto veio quebrado, não podendo ser montado. Além de extrema baixa qualidade nas peças. Veio todo descascado, com medidas diferentes, amassado. Solicito troca urgente por um produto em boas condico

[com filtro] openai/gpt-oss-120b
  TRECHO NA RESPOSTA: “baixa qualidade… péssima qualidade”  -> citado como ['R3']
  TEXTO REAL DE R3: baixa qualilidade. Esperava q a mercadoria tivesse mais qualidade ...pessima qualidade...nao indico a ninguem....

[com filtro] openai/gpt-oss-120b
  TRECHO NA RESPOSTA: “Só recebi um… não rastrear os 2 produtos”  -> citado como ['R20']
  TEXTO REAL DE R20: Não recomendo. Comprei 3 produtos dessa loja e só recebi um. Sendo que não tenho nem como rastrear onde foi parar meus 2 produ

In [6]:
# ===== Célula 10 — Salvar as respostas da comparação v2 em arquivo =====
import json

dados = [{"tipo": tipo, "modelo": modelo, "pergunta": pergunta, "filtros": filtros, "resposta": texto}
         for (tipo, modelo), (pergunta, filtros, texto) in respostas.items()]
arquivo = RAIZ / "resultados" / "comparacao_prompt_v2.json"
arquivo.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(dados)} respostas salvas em {arquivo}")

6 respostas salvas em c:\Users\domain\Tech Challenge - Fase 4\tech-challenge-fase4\resultados\comparacao_prompt_v2.json
